# TensorRT Model Optimization - Colab GPU Run

Run this notebook in Google Colab with a GPU runtime. It creates saved JSON artifacts for CUDA/TensorRT availability, PyTorch GPU latency, ONNX Runtime GPU or TensorRT latency, correctness, and comparable speedup review.

Do not use TensorRT speedup claims unless the selected provider is `TensorrtExecutionProvider` and the generated comparison report says the settings are comparable.

## 1. Runtime Setup

In Colab, select **Runtime > Change runtime type > GPU** before running. A T4 is enough for this small benchmark.

In [ ]:
!nvidia-smi
!python -m pip install -q --upgrade pip
!python -m pip install -q torch onnx onnxscript onnxruntime-gpu numpy

## 2. Clone Repo

This keeps the notebook reproducible even when opened directly from GitHub.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/nguyenthevietquang07/tensorrt-model-optimization.git"
WORKDIR = Path("/content/tensorrt-model-optimization")

if WORKDIR.exists():
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)

os.chdir(WORKDIR)
print(WORKDIR)

## 3. GPU Benchmark

This trains the same Iris MLP used by the local CPU path, exports ONNX, then benchmarks PyTorch CUDA against the fastest ONNX Runtime provider available in the runtime. Provider priority is TensorRT, then CUDA.

In [ ]:
import json
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch

sys.path.insert(0, str(WORKDIR))

from src.modelopt.comparison import compare_reports
from src.modelopt.real_data import fetch_iris_dataset, split_by_class
from src.modelopt.reporting import build_report
from src.modelopt.torch_onnx import (
    TorchOnnxConfig,
    compare_correctness,
    export_onnx_model,
    prepare_iris_tensors,
    train_iris_mlp,
)

REPORT_DIR = WORKDIR / "reports"
MODEL_DIR = WORKDIR / "models"
REPORT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

def write_json(relative_path, payload):
    path = WORKDIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    print(f"wrote {relative_path}")

def gpu_name():
    if not torch.cuda.is_available():
        return "no_cuda_gpu"
    return torch.cuda.get_device_name(0)

def synchronize():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Colab, choose Runtime > Change runtime type > GPU, then rerun.")

config = TorchOnnxConfig(epochs=160, trials=200, batch_size=30, onnx_path="models/iris_mlp_colab.onnx")
samples = fetch_iris_dataset()
train, test = split_by_class(samples)
data = prepare_iris_tensors(train, test)
model = train_iris_mlp(data, config)
onnx_path = export_onnx_model(model, data, WORKDIR / config.onnx_path)

hardware = gpu_name()
available_providers = ort.get_available_providers()
if "TensorrtExecutionProvider" in available_providers:
    provider = "TensorrtExecutionProvider"
    backend = "onnxruntime_tensorrt"
elif "CUDAExecutionProvider" in available_providers:
    provider = "CUDAExecutionProvider"
    backend = "onnxruntime_cuda"
else:
    raise RuntimeError(f"No ONNX Runtime GPU provider is available: {available_providers}")

device = torch.device("cuda")
model_gpu = model.to(device).eval()
batch_gpu = data.test_features[: config.batch_size].to(device)
durations_ms = []
checksum = 0.0
with torch.no_grad():
    for _ in range(20):
        _ = model_gpu(batch_gpu)
    synchronize()
    for _ in range(config.trials):
        start = time.perf_counter()
        logits = model_gpu(batch_gpu)
        synchronize()
        durations_ms.append((time.perf_counter() - start) * 1000)
        checksum += float(logits.sum().item())

pytorch_gpu_report = build_report(
    backend="pytorch_cuda_mlp",
    durations_ms=durations_ms,
    metadata={
        "hardware": hardware,
        "trials": config.trials,
        "vector_size": 4,
        "batch_size": config.batch_size,
        "model": "iris_mlp",
        "precision": "fp32",
        "checksum": round(checksum, 6),
    },
)

session = ort.InferenceSession(str(onnx_path), providers=[provider, "CPUExecutionProvider"])
batch_np = data.test_features[: config.batch_size].numpy().astype(np.float32)
durations_ms = []
checksum = 0.0
for _ in range(20):
    _ = session.run(["logits"], {"features": batch_np})[0]
for _ in range(config.trials):
    start = time.perf_counter()
    logits = session.run(["logits"], {"features": batch_np})[0]
    durations_ms.append((time.perf_counter() - start) * 1000)
    checksum += float(logits.sum())

candidate_report = build_report(
    backend=backend,
    durations_ms=durations_ms,
    metadata={
        "hardware": hardware,
        "trials": config.trials,
        "vector_size": 4,
        "batch_size": config.batch_size,
        "model": "iris_mlp",
        "precision": "fp32",
        "provider": provider,
        "checksum": round(checksum, 6),
    },
)

with torch.no_grad():
    torch_logits = model_gpu(batch_gpu).detach().cpu().numpy()
ort_logits = session.run(["logits"], {"features": batch_np})[0]
cpu_correctness = compare_correctness(model_gpu.cpu(), onnx_path, data)
max_abs_diff = float(np.max(np.abs(torch_logits - ort_logits)))
prediction_agreement = float((torch_logits.argmax(axis=1) == ort_logits.argmax(axis=1)).mean())
correctness_report = {
    "project": "tensorrt-model-optimization",
    "stage": "colab_gpu_correctness",
    "baseline_backend": "pytorch_cuda_mlp",
    "candidate_backend": backend,
    "provider": provider,
    "max_abs_diff": round(max_abs_diff, 8),
    "prediction_agreement": round(prediction_agreement, 6),
    "cpu_onnx_correctness_passed": bool(cpu_correctness["passed"]),
    "passed": max_abs_diff <= 1e-4 and prediction_agreement == 1.0,
    "claim_boundary": "Correctness compares PyTorch CUDA logits and the selected ONNX Runtime GPU/TensorRT provider on the same batch.",
}
comparison_report = compare_reports(pytorch_gpu_report, candidate_report)
environment_report = {
    "project": "tensorrt-model-optimization",
    "stage": "colab_gpu_environment",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "onnxruntime": ort.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": hardware,
    "available_onnxruntime_providers": available_providers,
    "selected_provider": provider,
    "tensorrt_provider_available": provider == "TensorrtExecutionProvider",
    "claim_boundary": "Environment report records provider availability. CUDA evidence is not TensorRT evidence unless selected_provider is TensorrtExecutionProvider.",
}

write_json("reports/colab_gpu_environment.json", environment_report)
write_json("reports/colab_gpu_pytorch_report.json", pytorch_gpu_report)
write_json("reports/colab_gpu_candidate_report.json", candidate_report)
write_json("reports/colab_gpu_correctness_report.json", correctness_report)
write_json("reports/colab_gpu_comparison_report.json", comparison_report)

summary = {
    "provider": provider,
    "hardware": hardware,
    "pytorch_mean_ms": pytorch_gpu_report["mean_ms"],
    "candidate_mean_ms": candidate_report["mean_ms"],
    "mean_speedup": comparison_report["mean_speedup"],
    "comparable_settings": comparison_report["comparable_settings"],
    "correctness_passed": correctness_report["passed"],
}
print(json.dumps(summary, indent=2, sort_keys=True))

## 4. Validate and Download Artifacts

Run the validator, then download the reports as a zip and place them back into the repo's `reports/` directory.

In [ ]:
!python scripts/validate_colab_gpu_artifacts.py
!zip -j colab_gpu_artifacts.zip reports/colab_gpu_environment.json reports/colab_gpu_pytorch_report.json reports/colab_gpu_candidate_report.json reports/colab_gpu_correctness_report.json reports/colab_gpu_comparison_report.json

from google.colab import files
files.download("colab_gpu_artifacts.zip")